# Scraping de Mercado Libre con Scrapling (anti-bloqueo)

**Qué cambió y por qué**

Mercado Libre empezó a bloquear el navegador automatizado de Selenium y a mandar a inicio de sesión / verificación, por eso el código anterior dejaba de traer productos. Esta versión reemplaza Selenium por **[Scrapling](https://github.com/D4Vinci/Scrapling)** y su `StealthyFetcher`, que usa un navegador *stealth* (Camoufox) diseñado para evadir la detección de bots y challenges tipo Cloudflare.

Además se eliminó la duplicación: antes la función de extracción (~200 líneas) estaba **repetida 12 veces**. Ahora está definida **una sola vez** y cada marca solo aporta su(s) URL(s).

**Cómo usarlo (primera vez)**
1. Ejecuta la celda de **instalación** (instala Scrapling y descarga el navegador stealth con `scrapling install`).
2. Ejecuta la celda de **configuración y funciones** (define todo el núcleo).
3. Ejecuta las celdas de cada **marca** (o todas de corrido).
4. Ejecuta la **consolidación** al final.

Si una marca sigue regresando vacío, vuelve a correr esa celda con `HEADLESS = False` (en la celda de configuración) para ver el navegador y, si aparece un challenge, déjalo resolver: con `solve_cloudflare=True` y `user_data_dir` la sesión se reutiliza entre corridas.


## 1. Configuración y funciones (núcleo compartido)

In [1]:

import re
import time
from datetime import datetime
import pandas as pd
from scrapling.fetchers import StealthyFetcher

# ============================ AJUSTES GLOBALES =============================
HEADLESS         = False   # Pon False para VER el navegador (útil si hay verificación/login)
NETWORK_IDLE     = True   # Espera a que la red quede inactiva (carga JS de la grilla)
SOLVE_CLOUDFLARE = True   # Intenta resolver challenges tipo Cloudflare automáticamente
GOOGLE_SEARCH    = True   # Simula llegar desde Google (ayuda contra anti-bot)
MAX_PAGES        = 20     # Máximo de páginas a seguir por URL semilla
WAIT_SELECTOR    = ".poly-card, .ui-search-result, .poly-component"  # espera a que carguen productos
# Carpeta para reutilizar la sesion/cookies entre corridas (como un perfil dedicado).
USER_DATA_DIR    = "scrapling_profile_ml"

# ============================ CONFIG DE MARCAS ============================
# marca (texto que va en la columna "Marca") -> lista de URLs semilla
BRANDS = {
    "SAMSUNG": {"urls": [
        "https://listado.mercadolibre.com.mx/tienda/samsung/_Container_ao-mobile-samsug#applied_filter_id%3Dofficial_store%26applied_filter_name%3DTiendas+oficiales%26applied_filter_order%3D2%26applied_value_id%3D295%26applied_value_name%3DSamsung%26applied_value_order%3D18%26applied_value_results%3D4%26is_custom%3Dfalse%26view_more_flag%3Dtrue",
    ], "paginate": True},
    "APPLE": {"urls": [
        "https://listado.mercadolibre.com.mx/_Tienda_apple_af_to_Container_ao-mobile-apple_NoIndex_True",
        "https://listado.mercadolibre.com.mx/_Desde_49_Tienda_apple_af_to_Container_ao-mobile-apple_NoIndex_True",
        "https://listado.mercadolibre.com.mx/_Desde_97_Tienda_apple_af_to_Container_ao-mobile-apple_NoIndex_True",
        "https://listado.mercadolibre.com.mx/_Desde_145_Tienda_apple_af_to_Container_ao-mobile-apple_NoIndex_True",
    ], "paginate": False},
    "XIAOMI": {"urls": [
        "https://listado.mercadolibre.com.mx/tienda/poco/_Container_ao-mobile-xiaomi#applied_filter_id%3Dofficial_store%26applied_filter_name%3DTiendas+oficiales%26applied_filter_order%3D3%26applied_value_id%3D3894%26applied_value_name%3DPOCO%26applied_value_order%3D10%26applied_value_results%3D6%26is_custom%3Dfalse",
        "https://listado.mercadolibre.com.mx/tienda/xiaomi/_Container_ao-mobile-xiaomi_NoIndex_True#applied_filter_id%3Dofficial_store%26applied_filter_name%3DTiendas+oficiales%26applied_filter_order%3D3%26applied_value_id%3D573%26applied_value_name%3DXiaomi%26applied_value_order%3D14%26applied_value_results%3D1%26is_custom%3Dfalse%26view_more_flag%3Dtrue",
    ], "paginate": False},
    "NUBIA": {"urls": [
        "https://listado.mercadolibre.com.mx/_Tienda_nubia_af_to_Container_ao-mobile-zte#applied_filter_id%3Dofficial_store%26applied_filter_name%3DTiendas+oficiales%26applied_filter_order%3D3%26applied_value_id%3D64116%26applied_value_name%3Dnubia%26applied_value_order%3D8%26applied_value_results%3D3%26is_custom%3Dfalse",
        "https://listado.mercadolibre.com.mx/_Tienda_nubia_af_to_Container_ao-mobile-nubia#applied_filter_id%3Dofficial_store%26applied_filter_name%3DTiendas+oficiales%26applied_filter_order%3D1%26applied_value_id%3D64116%26applied_value_name%3Dnubia%26applied_value_order%3D2%26applied_value_results%3D7%26is_custom%3Dfalse",
    ], "paginate": False},
    "ONEPLUS": {"urls": [
        "https://listado.mercadolibre.com.mx/_Tienda_oneplus_af_to_Container_ao-mobile-oneplus_NoIndex_True#applied_filter_id%3Dofficial_store%26applied_filter_name%3DTiendas+oficiales%26applied_filter_order%3D2%26applied_value_id%3D52307%26applied_value_name%3DONEPLUS%26applied_value_order%3D1%26applied_value_results%3D2%26is_custom%3Dfalse",
    ], "paginate": False},
    "CUBOT": {"urls": [
        "https://listado.mercadolibre.com.mx/_Tienda_cubot_af_to_Container_ao-mobile-cubot#applied_filter_id%3Dofficial_store%26applied_filter_name%3DTiendas+oficiales%26applied_filter_order%3D4%26applied_value_id%3D5776%26applied_value_name%3DCUBOT%26applied_value_order%3D1%26applied_value_results%3D18%26is_custom%3Dfalse",
    ], "paginate": False},
    "INFINIX": {"urls": [
        "https://listado.mercadolibre.com.mx/_Tienda_infinix_af_to_Container_ao-mobile-infinix#applied_filter_id%3Dofficial_store%26applied_filter_name%3DTiendas+oficiales%26applied_filter_order%3D2%26applied_value_id%3D6302%26applied_value_name%3DINFINIX%26applied_value_order%3D3%26applied_value_results%3D39%26is_custom%3Dfalse",
    ], "paginate": False},
    "HUAWEI": {"urls": [
        "https://listado.mercadolibre.com.mx/_Tienda_huawei_af_to_Container_ao-mobile-huawei#applied_filter_id%3Dofficial_store%26applied_filter_name%3DTiendas+oficiales%26applied_filter_order%3D3%26applied_value_id%3D2591%26applied_value_name%3DHuawei%26applied_value_order%3D4%26applied_value_results%3D8%26is_custom%3Dfalse",
    ], "paginate": False},
    "MOTOROLA": {"urls": [
        "https://listado.mercadolibre.com.mx/tienda/motorola/_Container_ao-mobile-motorola#applied_filter_id%3Dofficial_store%26applied_filter_name%3DTiendas+oficiales%26applied_filter_order%3D2%26applied_value_id%3D1466%26applied_value_name%3DMotorola%26applied_value_order%3D13%26applied_value_results%3D8%26is_custom%3Dfalse%26view_more_flag%3Dtrue",
    ], "paginate": True},
    "HONOR": {"urls": [
        "https://listado.mercadolibre.com.mx/tienda/honor/_Container_ao-mobile-honor#applied_filter_id%3Dofficial_store%26applied_filter_name%3DTiendas+oficiales%26applied_filter_order%3D2%26applied_value_id%3D2853%26applied_value_name%3DHONOR%26applied_value_order%3D7%26applied_value_results%3D3%26is_custom%3Dfalse",
    ], "paginate": False},
    "OPPO": {"urls": [
        "https://listado.mercadolibre.com.mx/tienda/oppo/#global_position=1",
    ], "paginate": True},
    "ZTE": {"urls": [
        "https://listado.mercadolibre.com.mx/tienda/mercado-libre/_Container_ao-mobile-zte_NoIndex_True#applied_filter_id%3Dofficial_store%26applied_filter_name%3DTiendas+oficiales%26applied_filter_order%3D3%26applied_value_id%3D4281%26applied_value_name%3DMercado+Libre%26applied_value_order%3D6%26applied_value_results%3D1%26is_custom%3Dfalse%26view_more_flag%3Dtrue",
        "https://listado.mercadolibre.com.mx/tienda/mercado-libre-electronica/_Container_ao-mobile-zte#applied_filter_id%3Dofficial_store%26applied_filter_name%3DTiendas+oficiales%26applied_filter_order%3D3%26applied_value_id%3D2856%26applied_value_name%3DMercado+Libre+Electronica%26applied_value_order%3D7%26applied_value_results%3D6%26is_custom%3Dfalse",
    ], "paginate": False},
}

# ============================== UTILIDADES ================================
def _first(node, css):
    if node is None:
        return None
    found = node.css(css)
    return found[0] if found else None

def _txt(node, css):
    el = _first(node, css)
    if el is None:
        return ""
    try:
        return el.get_all_text(strip=True)
    except Exception:
        return (el.text or "").strip()

import unicodedata

def _sin_acentos(s):
    return "".join(c for c in unicodedata.normalize("NFD", s)
                   if unicodedata.category(c) != "Mn")

# Lista SIN acentos (el nombre tambien se normaliza antes de comparar).
# Los colores compuestos van primero para que ganen sobre los simples.
COLORES = [
    'titanio negro', 'titanio blanco', 'titanio del desierto', 'titanio natural',
    'negro', 'black', 'blanco', 'white', 'azul', 'blue', 'rojo', 'red',
    'verde', 'green', 'amarillo', 'yellow', 'rosa', 'pink', 'morado', 'purple', 'purpura',
    'gris', 'gray', 'grey', 'dorado', 'gold', 'plateado', 'silver', 'plata', 'cian',
    'lila', 'grafito', 'lavanda', 'ultramarino', 'medianoche', 'cafe',
    'menta', 'multicolor', 'beige', 'violeta', 'naranja', 'midnight', 'titanio', 'starlight',
    'phantom', 'cream', 'lavender', 'mint'
]

def _limpiar_precio(precio):
    if precio and precio not in ("No discount", "No inventory"):
        cleaned = re.sub(r'[^\d.]', '', precio.replace(',', ''))
        try:
            return float(cleaned)
        except ValueError:
            return None
    return None

# ===================== EXTRACCION DE UN PRODUCTO =========================
def extraer_info_producto(product, marca, fecha_consulta):
    """Extrae info de una tarjeta de producto (Selector de Scrapling)."""
    try:
        nombre = ""
        for sel in (".poly-component__title", ".ui-search-item__title",
                    ".ui-search-item__title-label", "h2.ui-search-item__title",
                    "a.ui-search-link", ".poly-box .poly-component__title"):
            nombre = _txt(product, sel)
            if nombre:
                break

        color = ""
        if nombre:
            nl = _sin_acentos(nombre.lower())
            for c in COLORES:
                # limite de palabra: evita que 'red' matchee dentro de 'Redmi'
                if re.search(r'(?<![a-z])' + re.escape(c) + r'(?![a-z])', nl):
                    color = c.capitalize()
                    break

        capacidad = ""
        if nombre:
            m = re.search(r'(16|32|64|128|256|512|1)\s*(GB|TB)', nombre, re.IGNORECASE)
            if m:
                capacidad = f"{m.group(1)} {m.group(2).upper()}"

        url_producto = ""
        for sel in (".poly-component__title-wrapper a.poly-component__title",
                    "a.poly-component__title",
                    "a.poly-component__link.poly-component__link--carousel",
                    "a.poly-component__link", "a.ui-search-link",
                    "a.ui-search-item__group__element",
                    ".poly-component__title a", "h2.ui-search-item__title a"):
            link = _first(product, sel)
            if link is not None:
                href = link.attrib.get("href", "") or ""
                if href.startswith("http"):
                    url_producto = href; break
                elif href.startswith("/"):
                    url_producto = "https://www.mercadolibre.com.mx" + href; break

        sku = ""
        if url_producto:
            m = re.search(r'(MLM\d+)', url_producto)
            if m:
                sku = m.group(1)
        if not sku:
            sku = product.attrib.get("data-id") or product.attrib.get("data-item-id") or ""

        precio_original = precio_descuento = porcentaje_descuento = ""
        price_container = _first(product, ".poly-component__price")
        if price_container is not None:
            t = _txt(price_container, "s.andes-money-amount--previous .andes-money-amount__fraction")
            if t: precio_original = "$" + t
            t = _txt(price_container, ".poly-price__current .andes-money-amount__fraction")
            if t: precio_descuento = "$" + t
            t = _txt(price_container, ".andes-money-amount__discount")
            if t and "OFF" in t.upper():
                porcentaje_descuento = re.sub(r'(?i)off', '', t).strip()

        if not precio_original and not precio_descuento:
            for sel in (".andes-money-amount__fraction--previous",
                        ".ui-search-price__original-value",
                        ".price-tag-fraction--strikethrough",
                        ".ui-search-price__part--original"):
                t = _txt(product, sel)
                if t: precio_original = "$" + t; break
            for sel in (".andes-money-amount__fraction", ".price-tag-fraction"):
                el = _first(product, sel)
                if el is not None:
                    parent = el.parent
                    pcls = str(parent.attrib.get("class", "")) if parent is not None else ""
                    if not any(k in pcls for k in ("previous", "strikethrough", "original")):
                        precio_descuento = "$" + el.get_all_text(strip=True); break

        if not precio_original and precio_descuento:
            precio_original, precio_descuento = precio_descuento, "No discount"
        elif precio_original and not precio_descuento:
            precio_descuento = "No discount"
        elif not precio_original and not precio_descuento:
            precio_original, precio_descuento = "No inventory", "No discount"

        if not porcentaje_descuento:
            po, pd = _limpiar_precio(precio_original), _limpiar_precio(precio_descuento)
            porcentaje_descuento = f"{round(100*(po-pd)/po, 2)}%" if (po and pd and po > pd) else "No discount"

        return {"Marca": marca, "Nombre": nombre, "Color": color, "Capacidad": capacidad,
                "SKU": sku, "URL": url_producto, "FechaConsulta": fecha_consulta,
                "Precio_original": precio_original, "Precio_descuento": precio_descuento,
                "Porcentaje_descuento": porcentaje_descuento}
    except Exception as e:
        print(f"  Error al extraer producto: {e}")
        return None

# ===================== DESCARGA DE PAGINA (STEALTH) ======================
def fetch_page(url):
    """Descarga una pagina en un hilo con ProactorEventLoop (Windows + Jupyter)."""
    import asyncio
    import concurrent.futures
    def _fetch():
        loop = asyncio.ProactorEventLoop()
        asyncio.set_event_loop(loop)
        try:
            return StealthyFetcher.fetch(
                url,
                headless=HEADLESS,
                network_idle=NETWORK_IDLE,
                solve_cloudflare=SOLVE_CLOUDFLARE,
                google_search=GOOGLE_SEARCH,
                wait_selector=WAIT_SELECTOR,
                wait_selector_state="visible",
                user_data_dir=USER_DATA_DIR,
                timeout=60000,
            )
        finally:
            loop.close()
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(_fetch).result()

def _buscar_productos(page):
    for sel in (".ui-search-result", ".ui-search-results__item",
                ".poly-card", ".poly-component", "div[data-item-id]"):
        prods = page.css(sel)
        if prods:
            return prods
    return page.css('div[class*="search-result"], div[class*="poly"]')

def _siguiente_url(page):
    """Devuelve la URL de la pagina siguiente, o None si no hay / esta deshabilitada."""
    for sel in ("li.andes-pagination__button--next:not(.andes-pagination__button--disabled) a",
                ".andes-pagination__button--next a.andes-pagination__link",
                "a[title='Siguiente']"):
        found = page.css(sel)
        if found:
            href = found[0].attrib.get("href", "") or ""
            if href.startswith("http"):
                return href
    return None

# ===================== SCRAPER GENERICO POR MARCA ========================
def scrape_brand(marca, urls, paginate=True, max_pages=MAX_PAGES):
    print(f"\n=== Scrapeando {marca} ===")
    data, vistos = [], set()
    fecha_consulta = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for seed in urls:
        url, pagina = seed, 1
        while url and pagina <= max_pages:
            try:
                page = fetch_page(url)
            except Exception as e:
                print(f"  No se pudo descargar la pagina: {e}")
                print("  Sugerencia: corre '!scrapling install' o pon HEADLESS=False y reintenta.")
                break

            status = getattr(page, "status", None)
            productos = _buscar_productos(page)
            print(f"  Pagina {pagina} (HTTP {status}): {len(productos)} productos")

            for p in productos:
                info = extraer_info_producto(p, marca, fecha_consulta)
                if info and (info["Nombre"] or info["URL"]):
                    clave = info["SKU"] or info["URL"] or info["Nombre"]
                    if clave in vistos:
                        continue
                    vistos.add(clave)
                    data.append(info)

            if not paginate:
                break
            nxt = _siguiente_url(page)
            if not nxt:
                break
            url, pagina = nxt, pagina + 1
            time.sleep(1)

    print(f"  -> {marca}: {len(data)} productos unicos")
    return data

def scrape_and_export(marca):
    """Scrapea una marca de BRANDS y guarda su CSV del dia."""
    cfg = BRANDS[marca]
    data = scrape_brand(marca, cfg["urls"], paginate=cfg.get("paginate", True))
    if data:
        df = pd.DataFrame(data)
        fecha_hoy = datetime.now().strftime("%Y-%m-%d")
        nombre = f"mercadolibre_{marca.lower()}_{fecha_hoy}.csv"
        df.to_csv(nombre, index=False, encoding="utf-8-sig")
        print(f"  Guardado: {nombre}  ({len(df)} productos)")
        return df
    print(f"  No se encontraron productos para {marca}.")
    return None

print("Configuracion y funciones cargadas. Marcas disponibles:")
print(", ".join(BRANDS.keys()))

Configuracion y funciones cargadas. Marcas disponibles:
SAMSUNG, APPLE, XIAOMI, NUBIA, ONEPLUS, CUBOT, INFINIX, HUAWEI, MOTOROLA, HONOR, OPPO, ZTE


## 2. Scraping por marca
Cada celda usa el mismo núcleo; solo cambia la marca.

### Samsung

In [2]:
df_samsung = scrape_and_export("SAMSUNG")


=== Scrapeando SAMSUNG ===
  No se pudo descargar la pagina: 
  Sugerencia: corre '!scrapling install' o pon HEADLESS=False y reintenta.
  -> SAMSUNG: 0 productos unicos
  No se encontraron productos para SAMSUNG.


### Apple

In [3]:
df_apple = scrape_and_export("APPLE")


=== Scrapeando APPLE ===
  No se pudo descargar la pagina: 
  Sugerencia: corre '!scrapling install' o pon HEADLESS=False y reintenta.
  No se pudo descargar la pagina: 
  Sugerencia: corre '!scrapling install' o pon HEADLESS=False y reintenta.
  No se pudo descargar la pagina: 
  Sugerencia: corre '!scrapling install' o pon HEADLESS=False y reintenta.
  No se pudo descargar la pagina: 
  Sugerencia: corre '!scrapling install' o pon HEADLESS=False y reintenta.
  -> APPLE: 0 productos unicos
  No se encontraron productos para APPLE.


### Xiaomi

In [4]:
df_xiaomi = scrape_and_export("XIAOMI")


=== Scrapeando XIAOMI ===
  No se pudo descargar la pagina: 
  Sugerencia: corre '!scrapling install' o pon HEADLESS=False y reintenta.
  No se pudo descargar la pagina: 
  Sugerencia: corre '!scrapling install' o pon HEADLESS=False y reintenta.
  -> XIAOMI: 0 productos unicos
  No se encontraron productos para XIAOMI.


### Nubia

In [5]:
df_nubia = scrape_and_export("NUBIA")

Future exception was never retrieved
future: <Future finished exception=NotImplementedError()>
Traceback (most recent call last):
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736\1520370.py", line 257, in scrape_brand
    page = fetch_page(url)
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736\1520370.py", line 225, in fetch_page
    return pool.submit(_fetch).result()
           ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\_base.py", line 456, in result
    return self.__get_result()
           ~~~~~~~~~~~~~~~~~^^
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\_base.py", line 401, in __get_result
    raise self._exception
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\thread.py", line 59, in run
    result = self.fn(*self.args, **self.kwargs)
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736

Future exception was never retrieved
future: <Future finished exception=NotImplementedError()>
Traceback (most recent call last):
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736\1520370.py", line 257, in scrape_brand
    page = fetch_page(url)
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736\1520370.py", line 225, in fetch_page
    return pool.submit(_fetch).result()
           ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\_base.py", line 456, in result
    return self.__get_result()
           ~~~~~~~~~~~~~~~~~^^
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\_base.py", line 401, in __get_result
    raise self._exception
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\thread.py", line 59, in run
    result = self.fn(*self.args, **self.kwargs)
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736


=== Scrapeando NUBIA ===
  No se pudo descargar la pagina: 
  Sugerencia: corre '!scrapling install' o pon HEADLESS=False y reintenta.


Future exception was never retrieved
future: <Future finished exception=NotImplementedError()>
Traceback (most recent call last):
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736\1520370.py", line 257, in scrape_brand
    page = fetch_page(url)
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736\1520370.py", line 225, in fetch_page
    return pool.submit(_fetch).result()
           ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\_base.py", line 456, in result
    return self.__get_result()
           ~~~~~~~~~~~~~~~~~^^
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\_base.py", line 401, in __get_result
    raise self._exception
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\thread.py", line 59, in run
    result = self.fn(*self.args, **self.kwargs)
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736

  No se pudo descargar la pagina: 
  Sugerencia: corre '!scrapling install' o pon HEADLESS=False y reintenta.
  -> NUBIA: 0 productos unicos
  No se encontraron productos para NUBIA.


### Oneplus

In [6]:
df_oneplus = scrape_and_export("ONEPLUS")


=== Scrapeando ONEPLUS ===
  No se pudo descargar la pagina: 
  Sugerencia: corre '!scrapling install' o pon HEADLESS=False y reintenta.
  -> ONEPLUS: 0 productos unicos
  No se encontraron productos para ONEPLUS.


### Cubot

In [7]:
df_cubot = scrape_and_export("CUBOT")


=== Scrapeando CUBOT ===
  No se pudo descargar la pagina: 
  Sugerencia: corre '!scrapling install' o pon HEADLESS=False y reintenta.
  -> CUBOT: 0 productos unicos
  No se encontraron productos para CUBOT.


### Infinix

In [8]:
df_infinix = scrape_and_export("INFINIX")


=== Scrapeando INFINIX ===
  No se pudo descargar la pagina: 
  Sugerencia: corre '!scrapling install' o pon HEADLESS=False y reintenta.
  -> INFINIX: 0 productos unicos
  No se encontraron productos para INFINIX.


### Huawei

In [9]:
df_huawei = scrape_and_export("HUAWEI")


=== Scrapeando HUAWEI ===
  No se pudo descargar la pagina: 
  Sugerencia: corre '!scrapling install' o pon HEADLESS=False y reintenta.
  -> HUAWEI: 0 productos unicos
  No se encontraron productos para HUAWEI.


### Motorola

In [10]:
df_motorola = scrape_and_export("MOTOROLA")


=== Scrapeando MOTOROLA ===
  No se pudo descargar la pagina: 
  Sugerencia: corre '!scrapling install' o pon HEADLESS=False y reintenta.
  -> MOTOROLA: 0 productos unicos
  No se encontraron productos para MOTOROLA.


### Honor

In [11]:
df_honor = scrape_and_export("HONOR")


=== Scrapeando HONOR ===
  No se pudo descargar la pagina: 
  Sugerencia: corre '!scrapling install' o pon HEADLESS=False y reintenta.
  -> HONOR: 0 productos unicos
  No se encontraron productos para HONOR.


### Oppo

In [12]:
df_oppo = scrape_and_export("OPPO")


=== Scrapeando OPPO ===
  No se pudo descargar la pagina: 
  Sugerencia: corre '!scrapling install' o pon HEADLESS=False y reintenta.
  -> OPPO: 0 productos unicos
  No se encontraron productos para OPPO.


### ZTE

In [13]:
df_zte = scrape_and_export("ZTE")

Future exception was never retrieved
future: <Future finished exception=NotImplementedError()>
Traceback (most recent call last):
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736\1520370.py", line 257, in scrape_brand
    page = fetch_page(url)
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736\1520370.py", line 225, in fetch_page
    return pool.submit(_fetch).result()
           ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\_base.py", line 456, in result
    return self.__get_result()
           ~~~~~~~~~~~~~~~~~^^
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\_base.py", line 401, in __get_result
    raise self._exception
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\thread.py", line 59, in run
    result = self.fn(*self.args, **self.kwargs)
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736


=== Scrapeando ZTE ===


Future exception was never retrieved
future: <Future finished exception=NotImplementedError()>
Traceback (most recent call last):
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736\1520370.py", line 257, in scrape_brand
    page = fetch_page(url)
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736\1520370.py", line 225, in fetch_page
    return pool.submit(_fetch).result()
           ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\_base.py", line 456, in result
    return self.__get_result()
           ~~~~~~~~~~~~~~~~~^^
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\_base.py", line 401, in __get_result
    raise self._exception
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\thread.py", line 59, in run
    result = self.fn(*self.args, **self.kwargs)
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736

  No se pudo descargar la pagina: 
  Sugerencia: corre '!scrapling install' o pon HEADLESS=False y reintenta.
  No se pudo descargar la pagina: 
  Sugerencia: corre '!scrapling install' o pon HEADLESS=False y reintenta.
  -> ZTE: 0 productos unicos
  No se encontraron productos para ZTE.


## 4. Consolidación de todos los CSV del día

In [14]:
import pandas as pd
import glob
import os
from datetime import datetime

def consolidar_archivos_mercadolibre():
    """
    Consolida todos los archivos CSV de MercadoLibre en un solo archivo
    FILTRADOS POR FECHA DEL DÍA ACTUAL
    """
    print("Iniciando consolidación de archivos de MercadoLibre...")
    
    # Obtener la fecha actual para buscar archivos del día
    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    print(f"🗓️ Filtrando productos de la fecha: {fecha_hoy}")
    
    # Patrones de archivos a buscar (todos los archivos CSV de marcas)
    patrones_archivos = [
        f"mercadolibre_samsung_{fecha_hoy}.csv",
        f"mercadolibre_apple_{fecha_hoy}.csv",
        f"mercadolibre_xiaomi_{fecha_hoy}.csv",
        f"mercadolibre_nubia_{fecha_hoy}.csv",
        f"mercadolibre_oneplus_{fecha_hoy}.csv",
        f"mercadolibre_cubot_{fecha_hoy}.csv",
        f"mercadolibre_infinix_{fecha_hoy}.csv",
        f"mercadolibre_huawei_{fecha_hoy}.csv",
        f"mercadolibre_motorola_{fecha_hoy}.csv",
        f"mercadolibre_honor_{fecha_hoy}.csv",
        f"mercadolibre_oppo_{fecha_hoy}.csv",
        f"mercadolibre_zte_{fecha_hoy}.csv"
    ]
    
    # Lista para almacenar todos los DataFrames
    dataframes = []
    archivos_encontrados = []
    archivos_no_encontrados = []
    productos_filtrados_por_fecha = 0
    productos_total_antes_filtro = 0
    
    # Leer cada archivo si existe
    for patron in patrones_archivos:
        if os.path.exists(patron):
            try:
                df = pd.read_csv(patron, encoding='utf-8-sig')
                productos_total_antes_filtro += len(df)
                
                # NUEVO: Filtrar productos por fecha de consulta del día actual
                if 'FechaConsulta' in df.columns:
                    # Convertir la columna FechaConsulta a datetime
                    df['FechaConsulta_dt'] = pd.to_datetime(df['FechaConsulta'], errors='coerce')
                    
                    # Filtrar solo productos de hoy
                    fecha_hoy_dt = pd.to_datetime(fecha_hoy).date()
                    df_filtrado = df[df['FechaConsulta_dt'].dt.date == fecha_hoy_dt].copy()
                    
                    # Eliminar columna temporal
                    df_filtrado = df_filtrado.drop('FechaConsulta_dt', axis=1)
                    
                    productos_antes = len(df)
                    productos_despues = len(df_filtrado)
                    productos_filtrados = productos_antes - productos_despues
                    productos_filtrados_por_fecha += productos_filtrados
                    
                    if productos_despues > 0:
                        dataframes.append(df_filtrado)
                        archivos_encontrados.append(patron)
                        print(f"✓ Archivo leído: {patron} - {productos_despues}/{productos_antes} productos (filtrados {productos_filtrados} de fechas anteriores)")
                    else:
                        print(f"⚠️ Archivo {patron} no tiene productos de hoy - {productos_antes} productos de fechas anteriores")
                else:
                    # Si no tiene columna FechaConsulta, incluir todos
                    dataframes.append(df)
                    archivos_encontrados.append(patron)
                    print(f"✓ Archivo leído: {patron} - {len(df)} productos (sin filtro de fecha)")
                    
            except Exception as e:
                print(f"✗ Error al leer {patron}: {e}")
                archivos_no_encontrados.append(patron)
        else:
            archivos_no_encontrados.append(patron)
            print(f"✗ Archivo no encontrado: {patron}")
    
    # También buscar archivos con patrón general por si tienen fechas diferentes
    archivos_generales = glob.glob("mercadolibre_*_*.csv")
    for archivo in archivos_generales:
        if archivo not in archivos_encontrados and os.path.exists(archivo):
            try:
                df = pd.read_csv(archivo, encoding='utf-8-sig')
                productos_total_antes_filtro += len(df)
                
                # NUEVO: Filtrar productos por fecha de consulta del día actual
                if 'FechaConsulta' in df.columns:
                    # Convertir la columna FechaConsulta a datetime
                    df['FechaConsulta_dt'] = pd.to_datetime(df['FechaConsulta'], errors='coerce')
                    
                    # Filtrar solo productos de hoy
                    fecha_hoy_dt = pd.to_datetime(fecha_hoy).date()
                    df_filtrado = df[df['FechaConsulta_dt'].dt.date == fecha_hoy_dt].copy()
                    
                    # Eliminar columna temporal
                    df_filtrado = df_filtrado.drop('FechaConsulta_dt', axis=1)
                    
                    productos_antes = len(df)
                    productos_despues = len(df_filtrado)
                    productos_filtrados = productos_antes - productos_despues
                    productos_filtrados_por_fecha += productos_filtrados
                    
                    if productos_despues > 0:
                        dataframes.append(df_filtrado)
                        archivos_encontrados.append(archivo)
                        print(f"✓ Archivo adicional leído: {archivo} - {productos_despues}/{productos_antes} productos (filtrados {productos_filtrados} de fechas anteriores)")
                    else:
                        print(f"⚠️ Archivo adicional {archivo} no tiene productos de hoy - {productos_antes} productos de fechas anteriores")
                else:
                    # Si no tiene columna FechaConsulta, incluir todos
                    dataframes.append(df)
                    archivos_encontrados.append(archivo)
                    print(f"✓ Archivo adicional leído: {archivo} - {len(df)} productos (sin filtro de fecha)")
                    
            except Exception as e:
                print(f"✗ Error al leer {archivo}: {e}")
    
    if not dataframes:
        print("❌ No se encontraron archivos CSV de MercadoLibre con productos del día actual para consolidar.")
        print(f"📊 Total de productos encontrados en archivos: {productos_total_antes_filtro}")
        print(f"🗓️ Productos filtrados por fecha: {productos_filtrados_por_fecha}")
        return None
    
    # Consolidar todos los DataFrames
    print(f"\n📊 Consolidando {len(dataframes)} archivos...")
    df_consolidado = pd.concat(dataframes, ignore_index=True)
    
    # Agregar columna de fecha de consolidación
    df_consolidado['FechaConsolidacion'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # Reordenar columnas para mejor legibilidad
    columnas_ordenadas = [
        'Marca', 'Nombre', 'Color', 'Capacidad', 'SKU', 'URL',
        'FechaConsulta', 'Precio_original', 'Precio_descuento', 'Porcentaje_descuento',
        'FechaConsolidacion'
    ]
    
    # Verificar que todas las columnas existan
    columnas_existentes = [col for col in columnas_ordenadas if col in df_consolidado.columns]
    df_consolidado = df_consolidado[columnas_existentes]
    
    # Remover duplicados basados en SKU (si existe)
    productos_antes = len(df_consolidado)
    if 'SKU' in df_consolidado.columns:
        df_consolidado = df_consolidado.drop_duplicates(subset=['SKU'], keep='first')
        productos_despues = len(df_consolidado)
        duplicados_removidos = productos_antes - productos_despues
        if duplicados_removidos > 0:
            print(f"🔄 Removidos {duplicados_removidos} productos duplicados")
    
    # Ordenar por marca y nombre
    df_consolidado = df_consolidado.sort_values(['Marca', 'Nombre'], na_position='last')
    
    # Generar nombre del archivo consolidado con indicador de filtro por fecha
    nombre_archivo_consolidado = f"mercadolibre_consolidado_SOLO_{fecha_hoy}.csv"
    
    # Guardar archivo consolidado
    df_consolidado.to_csv(nombre_archivo_consolidado, index=False, encoding="utf-8-sig")
    
    # Mostrar resumen
    print(f"\n📈 RESUMEN DE CONSOLIDACIÓN (SOLO PRODUCTOS DEL DÍA ACTUAL):")
    print(f"🗓️ Fecha filtrada: {fecha_hoy}")
    print(f"✓ Archivos procesados: {len(archivos_encontrados)}")
    print(f"✗ Archivos no encontrados: {len(archivos_no_encontrados)}")
    print(f"📦 Productos totales encontrados: {productos_total_antes_filtro}")
    print(f"🗓️ Productos filtrados por fecha: {productos_filtrados_por_fecha}")
    print(f"📦 Productos consolidados del día actual: {len(df_consolidado)}")
    print(f"💾 Archivo guardado como: {nombre_archivo_consolidado}")
    
    # Mostrar estadísticas por marca
    print(f"\n📊 PRODUCTOS POR MARCA (SOLO DEL DÍA ACTUAL):")
    if 'Marca' in df_consolidado.columns:
        estadisticas_marca = df_consolidado['Marca'].value_counts().sort_values(ascending=False)
        for marca, cantidad in estadisticas_marca.items():
            print(f"  {marca}: {cantidad} productos")
    
    # Mostrar archivos procesados
    print(f"\n📁 ARCHIVOS PROCESADOS:")
    for archivo in archivos_encontrados:
        print(f"  ✓ {archivo}")
    
    if archivos_no_encontrados:
        print(f"\n❌ ARCHIVOS NO ENCONTRADOS:")
        for archivo in archivos_no_encontrados:
            print(f"  ✗ {archivo}")
    
    # Mostrar muestra de datos
    print(f"\n📋 MUESTRA DE DATOS CONSOLIDADOS (SOLO DEL DÍA ACTUAL):")
    print(df_consolidado.head(10))
    
    return df_consolidado

# Ejecutar la consolidación
print("🚀 INICIANDO CONSOLIDACIÓN CON FILTRO POR FECHA DEL DÍA ACTUAL")
fecha_hoy = datetime.now().strftime("%Y-%m-%d")
print(f"📅 Solo se incluirán productos con FechaConsulta del: {fecha_hoy}")

df_consolidado = consolidar_archivos_mercadolibre()

if df_consolidado is not None:
    print(f"\n🎉 ¡Consolidación completada exitosamente!")
    print(f"📅 Productos únicos del día actual ({fecha_hoy}): {len(df_consolidado)}")
    
    # Verificar rango de fechas en los datos consolidados
    if 'FechaConsulta' in df_consolidado.columns:
        fechas_unicas = df_consolidado['FechaConsulta'].apply(lambda x: str(x)[:10] if pd.notna(x) else 'Sin fecha').unique()
        print(f"🗓️ Fechas encontradas en datos consolidados: {list(fechas_unicas)}")
    
    # Opcional: Mostrar estadísticas adicionales
    if 'Precio_descuento' in df_consolidado.columns:
        # Limpiar precios para análisis
        def limpiar_precio_para_analisis(precio):
            if pd.isna(precio) or precio in ["No discount", "No inventory"]:
                return None
            try:
                return float(str(precio).replace('$', '').replace(',', ''))
            except:
                return None
        
        df_consolidado['Precio_numerico'] = df_consolidado['Precio_descuento'].apply(limpiar_precio_para_analisis)
        precios_validos = df_consolidado['Precio_numerico'].dropna()
        
        if len(precios_validos) > 0:
            print(f"\n💰 ESTADÍSTICAS DE PRECIOS (SOLO DEL DÍA ACTUAL):")
            print(f"  Precio promedio: ${precios_validos.mean():,.2f}")
            print(f"  Precio más bajo: ${precios_validos.min():,.2f}")
            print(f"  Precio más alto: ${precios_validos.max():,.2f}")
            print(f"  Productos con precio válido: {len(precios_validos)}")
        
        # Limpiar columna temporal
        df_consolidado = df_consolidado.drop('Precio_numerico', axis=1)
    
    print(f"\n📄 El archivo consolidado se guardó con nombre que incluye la fecha:")
    print(f"📁 mercadolibre_consolidado_SOLO_{fecha_hoy}.csv")
    
else:
    print("❌ No se pudo completar la consolidación.")
    print("🔍 Posibles causas:")
    print("  - No hay archivos CSV de MercadoLibre del día actual")
    print("  - Los archivos existen pero no tienen productos con FechaConsulta de hoy")
    print("  - Error al leer los archivos")

Future exception was never retrieved
future: <Future finished exception=NotImplementedError()>
Traceback (most recent call last):
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736\1520370.py", line 257, in scrape_brand
    page = fetch_page(url)
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736\1520370.py", line 225, in fetch_page
    return pool.submit(_fetch).result()
           ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\_base.py", line 456, in result
    return self.__get_result()
           ~~~~~~~~~~~~~~~~~^^
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\_base.py", line 401, in __get_result
    raise self._exception
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\thread.py", line 59, in run
    result = self.fn(*self.args, **self.kwargs)
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736

Future exception was never retrieved
future: <Future finished exception=NotImplementedError()>
Traceback (most recent call last):
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736\1520370.py", line 257, in scrape_brand
    page = fetch_page(url)
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736\1520370.py", line 225, in fetch_page
    return pool.submit(_fetch).result()
           ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\_base.py", line 456, in result
    return self.__get_result()
           ~~~~~~~~~~~~~~~~~^^
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\_base.py", line 401, in __get_result
    raise self._exception
  File "c:\Users\c50012218\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\thread.py", line 59, in run
    result = self.fn(*self.args, **self.kwargs)
  File "C:\Users\c50012218\AppData\Local\Temp\ipykernel_45736

🚀 INICIANDO CONSOLIDACIÓN CON FILTRO POR FECHA DEL DÍA ACTUAL
📅 Solo se incluirán productos con FechaConsulta del: 2026-06-14
Iniciando consolidación de archivos de MercadoLibre...
🗓️ Filtrando productos de la fecha: 2026-06-14
✗ Archivo no encontrado: mercadolibre_samsung_2026-06-14.csv
✗ Archivo no encontrado: mercadolibre_apple_2026-06-14.csv
✗ Archivo no encontrado: mercadolibre_xiaomi_2026-06-14.csv
✗ Archivo no encontrado: mercadolibre_nubia_2026-06-14.csv
✗ Archivo no encontrado: mercadolibre_oneplus_2026-06-14.csv
✗ Archivo no encontrado: mercadolibre_cubot_2026-06-14.csv
✗ Archivo no encontrado: mercadolibre_infinix_2026-06-14.csv
✗ Archivo no encontrado: mercadolibre_huawei_2026-06-14.csv
✗ Archivo no encontrado: mercadolibre_motorola_2026-06-14.csv
✗ Archivo no encontrado: mercadolibre_honor_2026-06-14.csv
✗ Archivo no encontrado: mercadolibre_oppo_2026-06-14.csv
✗ Archivo no encontrado: mercadolibre_zte_2026-06-14.csv
⚠️ Archivo adicional mercadolibre_apple_2026-02-05.csv n